# ChEMBL 37 chemical space (2.4 M × 64)

[ChEMBL 37](https://www.ebi.ac.uk/chembl/) is a manually curated database of bioactive
molecules with drug-like properties maintained by EMBL-EBI.  Each row is one compound,
represented as a 64-dimensional TruncatedSVD projection of a 2048-bit Morgan fingerprint
(radius 2) computed with [RDKit](https://www.rdkit.org/).

**Licence:** ChEMBL data is released under
[CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).
RDKit is BSD-licensed.  scikit-learn is BSD-licensed.

**Data source:** `chembl_37_chemreps.txt.gz` from the
[ChEMBL FTP site](https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/)
(292 MB compressed, ~2.4 M compounds).  Compound names and clinical-phase metadata
are fetched for named compounds only via the
[ChEMBL REST API](https://www.ebi.ac.uk/chembl/api/data/).
Structure images use the ChEMBL image API (PNG format).

**Image decision:** compound structure thumbnails use the PNG endpoint
(`…/api/data/image/<id>.png`) rather than SVG.  PNG rasterises reliably in every
Atlas code path — hover card *and* annotation overlay — without any special handling.
The SVG endpoint (`…/<id>.svg`) works in the hover card but can yield zero-dimension
thumbnails in the annotation overlay's canvas rasterisation path.

**First run:** downloading takes a few minutes; fingerprint computation and SVD
projection for all 2.4 M compounds takes roughly 5–15 minutes depending on hardware.
The final `chembl.parquet` is cached in `~/.cache/positron-atlas/datasets/chembl/`.
Subsequent runs skip every step that is already cached.

**Quick try:** set `SAMPLE = True` to process only the first 100,000 compounds
(~2–3 minutes on most machines; fingerprinting is fast but the metadata API
still takes ~1 minute for a small subset of named compounds).

**What to try in Atlas:**
- Hover a point to see the compound name, structure image (`image_url` column) and
  a link to the ChEMBL compound page.
- Color by `phase` to highlight clinical-stage compounds (Approved = phase 4 drugs).
- Color by `mw` (histogram filter) to explore molecular weight.
- Search for a drug name such as *aspirin*, *imatinib*, or *sildenafil*.
- Turn on cluster labels to discover structural families.
- Annotate a cluster of approved CNS drugs or kinase inhibitors.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set SAMPLE = True to process only the first 100,000 compounds (~2–3 min first
# run: fingerprinting is fast; metadata API still fetches ~5,000 named compounds).
# Set SAMPLE = False for the full 2.4 M compounds (~15–30 min first run).
SAMPLE = False

CHEMREPS_URL = "https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_37_chemreps.txt.gz"
N_COMPONENTS = 64        # TruncatedSVD output dimensions (f0 … f63)
CHUNK_SIZE   = 50_000    # compounds per fingerprint / projection chunk
SVD_SAMPLE   = 200_000   # compounds used to fit TruncatedSVD
SAMPLE_N     = 100_000   # compounds to process in SAMPLE mode
SAMPLE_META_PAGES = 5    # API pages (1 k/page) to fetch in SAMPLE mode

In [ ]:
import csv, gzip, json, time, urllib.request
from pathlib import Path

import numpy as np
import polars as pl
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
from scipy.sparse import lil_matrix
from sklearn.decomposition import TruncatedSVD
import pyarrow as pa
import pyarrow.parquet as pq

CACHE = Path.home() / ".cache" / "positron-atlas" / "datasets" / "chembl"
CHEMREPS_PATH = CACHE / "chembl_37_chemreps.txt.gz"
SVD_CACHE     = CACHE / "svd_components.npz"
FEATURES_PATH = CACHE / "chembl_features.parquet"
META_PATH     = CACHE / "chembl_meta.parquet"
CHEMBL_PATH   = CACHE / "chembl.parquet"
CACHE.mkdir(parents=True, exist_ok=True)


def _download(url: str, path: Path) -> Path:
    """Download url to path atomically; skip if already present."""
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(path)
    return path


def _mol_to_fp(mol) -> np.ndarray:
    """Return 2048-bit Morgan (radius-2) fingerprint as uint8 array."""
    arr = np.zeros(2048, dtype=np.uint8)
    if mol is not None:
        DataStructs.ConvertToNumpyArray(
            AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048), arr,
        )
    return arr

In [ ]:
# ── Phase 1: download chemreps ────────────────────────────────────────────────
if CHEMBL_PATH.exists():
    print("chembl.parquet already cached — skipping all phases")
elif CHEMREPS_PATH.exists():
    print(f"chemreps already downloaded ({CHEMREPS_PATH.stat().st_size / 1e9:.2f} GB)")
else:
    _t0 = time.perf_counter()
    _download(CHEMREPS_URL, CHEMREPS_PATH)
    print(f"Downloaded in {time.perf_counter() - _t0:.0f} s  ({CHEMREPS_PATH.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# ── Phase 2: fit TruncatedSVD on a sample of Morgan fingerprints ─────────────
# TruncatedSVD (no mean-centering) is appropriate for bit matrices.
# We build a scipy sparse matrix for the sample to keep memory under control
# (~100 bits set per fingerprint → ~95 % sparsity).

if CHEMBL_PATH.exists():
    print("chembl.parquet already cached — skipping Phase 2")
elif SVD_CACHE.exists():
    _npz = np.load(SVD_CACHE)
    _svd_components = _npz["components"]          # shape (N_COMPONENTS, 2048)
    print(f"Loaded SVD cache: {_svd_components.shape}")
else:
    _n_target = min(SVD_SAMPLE, SAMPLE_N if SAMPLE else SVD_SAMPLE)
    print(f"Phase 2: reading up to {_n_target:,} fingerprints for SVD fitting...")
    _t0 = time.perf_counter()
    _sample_fps: list[np.ndarray] = []

    with gzip.open(CHEMREPS_PATH, "rt") as _f:
        _reader = csv.DictReader(_f, delimiter="	")
        for _row in _reader:
            if len(_sample_fps) >= _n_target:
                break
            _mol = Chem.MolFromSmiles(_row["canonical_smiles"] or "")
            _sample_fps.append(_mol_to_fp(_mol))

    print(f"  Read {len(_sample_fps):,} fingerprints in {time.perf_counter() - _t0:.0f} s; building sparse matrix...")
    _t1 = time.perf_counter()
    _sparse = lil_matrix((len(_sample_fps), 2048), dtype=np.float32)
    for _i, _fp in enumerate(_sample_fps):
        _nz = _fp.nonzero()[0]
        if _nz.size:
            _sparse[_i, _nz] = 1.0
    _X = _sparse.tocsr()
    del _sparse, _sample_fps

    print(f"  Sparse matrix ({_X.shape}) built in {time.perf_counter() - _t1:.0f} s; fitting TruncatedSVD({N_COMPONENTS})...")
    _svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
    _svd.fit(_X)
    del _X

    _svd_components = _svd.components_.astype(np.float32)   # (64, 2048)
    np.savez(SVD_CACHE, components=_svd_components)
    print(f"  SVD fitted and cached in {time.perf_counter() - _t0:.0f} s — {_svd_components.shape}")

In [ ]:
# ── Phase 3: compute fingerprints and MW for all compounds, project to 64 dims ─
# Processes CHUNK_SIZE compounds at a time; total memory stays under ~1 GB.

if CHEMBL_PATH.exists():
    print("chembl.parquet already cached — skipping Phase 3")
elif FEATURES_PATH.exists():
    print("Features parquet already cached")
else:
    print("Phase 3: fingerprinting all compounds...")
    _t0 = time.perf_counter()
    _all_ids: list[str] = []
    _all_mws: list[float | None] = []
    _all_feats_parts: list[np.ndarray] = []
    _chunk_ids: list[str] = []
    _chunk_mws: list[float | None] = []
    _chunk_fps: list[np.ndarray] = []
    _n_total = _n_invalid = 0

    def _flush_chunk() -> None:
        _X = np.stack(_chunk_fps, axis=0).astype(np.float32)
        _all_feats_parts.append(_X @ _svd_components.T)
        _all_ids.extend(_chunk_ids)
        _all_mws.extend(_chunk_mws)
        _chunk_ids.clear(); _chunk_mws.clear(); _chunk_fps.clear()

    with gzip.open(CHEMREPS_PATH, "rt") as _f:
        _reader = csv.DictReader(_f, delimiter="	")
        for _row in _reader:
            if SAMPLE and _n_total >= SAMPLE_N:
                break
            _mol = Chem.MolFromSmiles(_row["canonical_smiles"] or "")
            if _mol is None:
                _n_invalid += 1
            _chunk_ids.append(_row["chembl_id"])
            _chunk_mws.append(round(Descriptors.MolWt(_mol), 2) if _mol else None)
            _chunk_fps.append(_mol_to_fp(_mol))
            _n_total += 1
            if len(_chunk_fps) >= CHUNK_SIZE:
                _flush_chunk()
                _elapsed = time.perf_counter() - _t0
                print(f"  {_n_total:,} compounds at {_n_total / _elapsed:,.0f}/s")

    if _chunk_fps:
        _flush_chunk()

    _all_feats = np.concatenate(_all_feats_parts, axis=0).astype(np.float32)
    print(
        f"Phase 3 done: {len(_all_ids):,} compounds ({_n_invalid:,} invalid SMILES) "
        f"in {time.perf_counter() - _t0:.0f} s"
    )

    _cols: dict = {"chembl_id": _all_ids, "mw": pa.array(_all_mws, type=pa.float32())}
    for _i in range(N_COMPONENTS):
        _cols[f"f{_i}"] = _all_feats[:, _i]
    pq.write_table(pa.table(_cols), FEATURES_PATH)
    print(f"Saved features parquet ({FEATURES_PATH.stat().st_size / 1e6:.0f} MB)")
    del _all_feats, _all_feats_parts

In [ ]:
# ── Phase 4: fetch pref_name and max_phase for named compounds via REST API ────
# ~49,000 named compounds out of 2.4 M; the rest fall back to chembl_id as title.
# In SAMPLE mode only SAMPLE_META_PAGES × 1,000 compounds are fetched to stay fast.

if CHEMBL_PATH.exists():
    print("chembl.parquet already cached — skipping Phase 4")
elif META_PATH.exists():
    print("Metadata parquet already cached")
else:
    print("Phase 4: fetching named-compound metadata from ChEMBL API...")
    _t0 = time.perf_counter()
    _meta_ids: list[str] = []
    _meta_names: list[str | None] = []
    _meta_phases: list[float | None] = []
    _offset = 0
    _limit  = 1000
    _total  = None
    _page   = 0

    while True:
        _url = (
            "https://www.ebi.ac.uk/chembl/api/data/molecule.json"
            f"?pref_name__isnull=False&limit={_limit}&offset={_offset}"
            "&fields=molecule_chembl_id,pref_name,max_phase"
        )
        _req = urllib.request.Request(_url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(_req, timeout=30) as _resp:
            _data = json.loads(_resp.read())

        for _m in _data["molecules"]:
            _meta_ids.append(_m["molecule_chembl_id"])
            _meta_names.append(_m.get("pref_name"))
            _phase = _m.get("max_phase")
            _meta_phases.append(float(_phase) if _phase is not None else None)

        if _total is None:
            _total = _data["page_meta"]["total_count"]
            print(f"  Total named compounds: {_total:,}")

        _page   += 1
        _offset += _limit
        if _offset >= _total:
            break
        if SAMPLE and _page >= SAMPLE_META_PAGES:
            print(f"  SAMPLE mode: stopping after {_page} pages ({len(_meta_ids):,} compounds)")
            break
        if _offset % 5_000 == 0:
            print(f"  {_offset:,}/{_total:,} ...")

    pq.write_table(
        pa.table({
            "chembl_id":  _meta_ids,
            "pref_name":  pa.array(_meta_names, type=pa.string()),
            "max_phase":  pa.array(_meta_phases, type=pa.float32()),
        }),
        META_PATH,
    )
    print(f"Phase 4 done: {len(_meta_ids):,} named compounds in {time.perf_counter() - _t0:.0f} s")

In [ ]:
# ── Phase 5: join features and metadata, write chembl.parquet ─────────────────
if CHEMBL_PATH.exists():
    print("chembl.parquet already cached — loading")
else:
    print("Phase 5: joining features and metadata...")
    _feats = pl.read_parquet(FEATURES_PATH)
    _meta  = pl.read_parquet(META_PATH)

    # Map numeric max_phase to a human-readable string so Atlas treats it as a
    # categorical colour column rather than a numeric feature (avoids null-drops).
    chembl = (
        _feats
        .join(_meta, on="chembl_id", how="left")
        .with_columns(
            pl.coalesce(["pref_name", "chembl_id"]).alias("title"),
            pl.format(
                "https://www.ebi.ac.uk/chembl/compound_report_card/{}/",
                pl.col("chembl_id"),
            ).alias("url"),
            pl.format(
                "https://www.ebi.ac.uk/chembl/api/data/image/{}.png",
                pl.col("chembl_id"),
            ).alias("image_url"),
            (
                pl.when(pl.col("max_phase") == 4.0).then(pl.lit("Approved"))
                .when(pl.col("max_phase") == 3.0).then(pl.lit("Phase 3"))
                .when(pl.col("max_phase") == 2.0).then(pl.lit("Phase 2"))
                .when(pl.col("max_phase") == 1.0).then(pl.lit("Phase 1"))
                .when(pl.col("max_phase") == 0.5).then(pl.lit("Early Phase 1"))
                .when(pl.col("max_phase") == -1.0).then(pl.lit("Withdrawn"))
                .otherwise(pl.lit(None, dtype=pl.String))
            ).alias("phase"),
        )
        .drop("pref_name", "max_phase")
        .select(
            ["chembl_id", "title", "url", "image_url", "phase", "mw"]
            + [f"f{i}" for i in range(N_COMPONENTS)]
        )
    )
    chembl.write_parquet(CHEMBL_PATH)
    print(f"Saved chembl.parquet ({CHEMBL_PATH.stat().st_size / 1e9:.2f} GB, {len(chembl):,} rows)")

chembl = pl.read_parquet(CHEMBL_PATH)
chembl.shape

## Open it in Atlas

1. In the **Variables** pane, double-click **`chembl`** to open it in the Data Explorer.
2. Click **Open in Atlas**, the first button in the Data Explorer's toolbar (top left).

Or run **Atlas: Open DataFrame…** from the Command Palette and pick `chembl`.

## Things to try

- **Color by `phase`.**  Approved drugs form a distinctive population; most compounds
  have no clinical-phase data (shown as missing).
- **Color by `mw`** and use the histogram filter to zoom in on drug-like compounds
  (MW 150–500).  The Lipinski "rule of five" space becomes visible.
- **Hover** a compound: the structure image is loaded from the ChEMBL image API (PNG).
  Set `image_url` as the **Image URL column** in the **Tooltip** tab if Atlas does not
  auto-detect it.
- **Cmd/Ctrl-click** (when `url` is the link column) to open the compound card on
  the ChEMBL website.
- **Search** for *aspirin*, *imatinib*, or *sildenafil* to find approved drugs and see
  where they sit in chemical space.
- **Annotate** the imatinib cluster (kinase inhibitors) or aspirin / NSAID cluster.